In [1]:
from __future__ import annotations

import hashlib
import json
import math
import platform
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import torch
from sklearn.ensemble import IsolationForest
from sklearn.exceptions import InconsistentVersionWarning
from sklearn.neighbors import LocalOutlierFactor
from torch import nn

In [8]:
# ================== 01. CẤU HÌNH PHƯƠNG ÁN A ==================
# Nếu notebook và outputs ở Stocks/notebooks/, để None là đủ.
# Nếu VS Code chạy kernel ở thư mục khác, có thể sửa DUY NHẤT dòng này:
# ROOT_OVERRIDE = Path(r'D:\LUẬN VĂN\stock-news-anomaly-project\Stocks\notebooks')
ROOT_OVERRIDE = None


def choose_root():
    if ROOT_OVERRIDE is not None:
        chosen = Path(ROOT_OVERRIDE).expanduser().resolve()
        if not (chosen / 'outputs' / '03_feature_engineering' / '03_model_ready_development.csv').is_file():
            raise FileNotFoundError(
                'ROOT_OVERRIDE chưa trỏ vào Stocks/notebooks/ chứa thư mục outputs. '
                f'Đang chọn: {chosen}'
            )
        return chosen

    cwd = Path.cwd().resolve()
    candidates = []
    for base in [cwd, *list(cwd.parents)[:4]]:
        candidates.extend([base, base / 'notebooks', base / 'Stocks' / 'notebooks'])
    for candidate in dict.fromkeys(candidates):
        if (candidate / 'outputs' / '03_feature_engineering' / '03_model_ready_development.csv').is_file():
            return candidate
    raise FileNotFoundError(
        'Không tự tìm thấy thư mục Stocks/notebooks/ chứa outputs. '
        f'Python hiện chạy tại: {cwd}. Hãy đặt ROOT_OVERRIDE ở đầu ô cấu hình.'
    )

ROOT = choose_root()
OUTPUT_DIR = ROOT / 'outputs' / '07_threshold_calibration_A'
FEATURES = ['z_gap', 'z_intraday', 'z_range', 'z_volume']
MODEL_NAMES = ['IF', 'AE', 'LOF']
QUANTILE = 0.99
RULE = 'score > threshold'
EXPECTED_DEV_SHA = '0f035c54869b7389548ce6ce0b574335ed149e8378b09351eaa8245b840f472a'
EXPECTED_CAL_N = 962
EXPECTED_VAL_SELECT_N = 1430
EXPECTED_TICKERS = 13
EXPECTED_CAL_DAYS = 74
DATE_START = '2025-09-11'
DATE_END = '2025-12-23'


def find_required(label, candidates):
    locations = [ROOT / name for name in candidates]
    for path in locations:
        if path.is_file():
            return path
    raise FileNotFoundError(
        f'Không tìm thấy file {label}. Thư mục làm việc ROOT={ROOT}\n'
        + '\n'.join(str(p) for p in locations)
    )


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


PATHS = {
    'dev': find_required('03_model_ready_development.csv', [
        'outputs/03_feature_engineering/03_model_ready_development.csv',
        '03_model_ready_development.csv'
    ]),
    'features_all': find_required('03_features_all.csv (chỉ checksum)', [
        'outputs/03_feature_engineering/03_features_all.csv',
        '03_features_all.csv'
    ]),
    'feature_config': find_required('03_feature_config.json', [
        'outputs/03_feature_engineering/03_feature_config.json', '03_feature_config.json'
    ]),
    'if': find_required('IF đã huấn luyện Bước 05', [
        'outputs/05_model_training_A/05_if.joblib', '05_if.joblib'
    ]),
    'ae': find_required('AE checkpoint tốt nhất Bước 06.2', [
        'outputs/06_2_ae_validation_A/06_2_ae_seed42_lr001_epoch1495_best.pt',
        '06_2_ae_seed42_lr001_epoch1495_best.pt'
    ]),
    'lof': find_required('LOF đã chốt Bước 06.3', [
        'outputs/06_3_lof_validation_A/06_3_lof_baseline_k20_euclidean.joblib',
        '06_3_lof_baseline_k20_euclidean.joblib'
    ]),
    'baseline_06_5': find_required('06_5_aligned_val_scores.csv', [
        'outputs/06_5_model_comparison_A/06_5_aligned_val_scores.csv',
        '06_5_aligned_val_scores.csv'
    ]),
}

assert sha256_file(PATHS['dev']) == EXPECTED_DEV_SHA, (
    'File development không đúng phiên bản đã dùng ở Bước 05–06.5. '
    'Dừng để không tính ngưỡng sai.'
)
with PATHS['feature_config'].open(encoding='utf-8') as fh:
    feature_config = json.load(fh)
assert feature_config['window_prior_valid_feature_values_per_ticker'] == 60
assert feature_config['features_for_model'] == FEATURES
assert abs(feature_config['train_floor_quantile_positive_mad_scales'] - 0.1) < 1e-12
assert set(feature_config['train_floors']) == {'gap', 'intraday', 'range', 'volume'}

print('ROOT:', ROOT)
print('PASS 01 — W=60, floor Train gốc, 4 đặc trưng cố định, checkpoint đầu vào đầy đủ.')

ROOT: D:\LUẬN VĂN\stock-news-anomaly-project\Stocks\notebooks
PASS 01 — W=60, floor Train gốc, 4 đặc trưng cố định, checkpoint đầu vào đầy đủ.


In [3]:
torch.set_num_threads(1)

class SmallAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(4,2),nn.Tanh(),nn.Linear(2,4))
    def forward(self, x):
        return self.net(x)

@torch.no_grad()
def score_ae(model, x):
    t = torch.from_numpy(np.array(x,dtype=np.float32,copy=True))
    model.eval()
    return (model(t)-t).square().mean(dim=1).cpu().numpy().astype(float)

version_warnings=[]
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always',InconsistentVersionWarning)
    model_if = joblib.load(PATHS['if'])
    model_lof = joblib.load(PATHS['lof'])
    version_warnings = [str(w.message) for w in caught if issubclass(w.category,InconsistentVersionWarning)]

assert isinstance(model_if, IsolationForest)
assert model_if.n_features_in_ == 4
assert model_if.n_estimators == 200 and model_if.max_samples == 256
assert model_if.random_state == 42 and model_if.max_features == 1.0
assert model_if.contamination == 'auto'
assert isinstance(model_lof, LocalOutlierFactor)
assert model_lof.n_features_in_ == 4
assert model_lof.n_neighbors == 20 and model_lof.novelty is True
assert model_lof.metric == 'minkowski' and model_lof.p == 2

ckpt = torch.load(PATHS['ae'],map_location='cpu',weights_only=True)
assert ckpt['seed'] == 42 and ckpt['best_epoch'] == 1495
assert list(ckpt['features']) == FEATURES
assert ckpt['architecture'] == '4-2-4'
assert abs(ckpt['learning_rate'] - 0.001) <= 1e-10
model_ae = SmallAE()
model_ae.load_state_dict(ckpt['state_dict'],strict=True)
model_ae.eval()


def score_three(x):
    # Điểm CÀNG CAO => càng khác thường; không dùng predict() mặc định
    return {
        'IF': -model_if.score_samples(x),
        'AE': score_ae(model_ae,x),
        'LOF': -model_lof.score_samples(x),
    }

if version_warnings:
    print('CẢNH BÁO: phiên bản scikit-learn hiện tại khác bản lưu model:')
    print('  ',version_warnings[0])
    print('Sẽ đối chiếu bắt buộc điểm Bước 06.5 trước khi hiệu chỉnh.')
print('Đã tải checkpoint IF 05, AE 06.2 epoch1495, LOF 06.3 k20. Không train lại.')

Đã tải checkpoint IF 05, AE 06.2 epoch1495, LOF 06.3 k20. Không train lại.


In [9]:
# ================== 02. NẠP HAI TẬP VALIDATION ==================
DEV_COLS = ['ticker', 'date', 'split', 'val_role', *FEATURES]
dev = pd.read_csv(PATHS['dev'], encoding='utf-8-sig', usecols=DEV_COLS)
assert len(dev) == 12_779
assert not dev['split'].eq('test').any(), 'Cấm có dữ liệu Test trong file development'
assert (dev['split'] == 'train').sum() == 10_387
assert not dev.duplicated(['ticker', 'date']).any()
dev['date'] = pd.to_datetime(dev['date'], format='%Y-%m-%d', errors='raise')
assert np.isfinite(dev[FEATURES].to_numpy(dtype=float)).all()

select = dev.loc[
    dev['split'].eq('val') & dev['val_role'].eq('select'),
    ['ticker', 'date', *FEATURES]
].copy().sort_values(['date', 'ticker']).reset_index(drop=True)

cal = dev.loc[
    dev['split'].eq('val') & dev['val_role'].eq('calibrate'),
    ['ticker', 'date', *FEATURES]
].copy().sort_values(['date', 'ticker']).reset_index(drop=True)

assert len(select) == EXPECTED_VAL_SELECT_N
assert len(cal) == EXPECTED_CAL_N
assert not select.duplicated(['ticker', 'date']).any()
assert not cal.duplicated(['ticker', 'date']).any()
assert select['date'].max() < cal['date'].min()
assert cal['ticker'].nunique() == EXPECTED_TICKERS
assert cal['date'].nunique() == EXPECTED_CAL_DAYS
assert cal['date'].min().strftime('%Y-%m-%d') == DATE_START
assert cal['date'].max().strftime('%Y-%m-%d') == DATE_END
assert cal.groupby('ticker').size().eq(EXPECTED_CAL_DAYS).all()

X_select = np.array(select[FEATURES], dtype=np.float32, copy=True, order='C')
X_cal = np.array(cal[FEATURES], dtype=np.float32, copy=True, order='C')
assert X_select.shape == (1430, 4) and X_cal.shape == (962, 4)

print(f'PASS 02 — Validation-select: {len(select):,} phiên (kiểm checkpoint).')
print(f'PASS 02 — Validation-calibrate: {len(cal):,} phiên, {EXPECTED_CAL_DAYS} ngày, {EXPECTED_TICKERS} mã.')

PASS 02 — Validation-select: 1,430 phiên (kiểm checkpoint).
PASS 02 — Validation-calibrate: 962 phiên, 74 ngày, 13 mã.


In [10]:
torch.set_num_threads(1)

class SmallAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(4,2),nn.Tanh(),nn.Linear(2,4))
    def forward(self, x):
        return self.net(x)

@torch.no_grad()
def score_ae(model, x):
    t = torch.from_numpy(np.array(x,dtype=np.float32,copy=True))
    model.eval()
    return (model(t)-t).square().mean(dim=1).cpu().numpy().astype(float)

version_warnings=[]
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always',InconsistentVersionWarning)
    model_if = joblib.load(PATHS['if'])
    model_lof = joblib.load(PATHS['lof'])
    version_warnings = [str(w.message) for w in caught if issubclass(w.category,InconsistentVersionWarning)]

assert isinstance(model_if, IsolationForest)
assert model_if.n_features_in_ == 4
assert model_if.n_estimators == 200 and model_if.max_samples == 256
assert model_if.random_state == 42 and model_if.max_features == 1.0
assert model_if.contamination == 'auto'
assert isinstance(model_lof, LocalOutlierFactor)
assert model_lof.n_features_in_ == 4
assert model_lof.n_neighbors == 20 and model_lof.novelty is True
assert model_lof.metric == 'minkowski' and model_lof.p == 2

ckpt = torch.load(PATHS['ae'],map_location='cpu',weights_only=True)
assert ckpt['seed'] == 42 and ckpt['best_epoch'] == 1495
assert list(ckpt['features']) == FEATURES
assert ckpt['architecture'] == '4-2-4'
assert abs(ckpt['learning_rate'] - 0.001) <= 1e-10
model_ae = SmallAE()
model_ae.load_state_dict(ckpt['state_dict'],strict=True)
model_ae.eval()


def score_three(x):
    # Điểm CÀNG CAO => càng khác thường; không dùng predict() mặc định
    return {
        'IF': -model_if.score_samples(x),
        'AE': score_ae(model_ae,x),
        'LOF': -model_lof.score_samples(x),
    }

if version_warnings:
    print('CẢNH BÁO: phiên bản scikit-learn hiện tại khác bản lưu model:')
    print('  ',version_warnings[0])
    print('Sẽ đối chiếu bắt buộc điểm Bước 06.5 trước khi hiệu chỉnh.')
print('Đã tải checkpoint IF 05, AE 06.2 epoch1495, LOF 06.3 k20. Không train lại.')

Đã tải checkpoint IF 05, AE 06.2 epoch1495, LOF 06.3 k20. Không train lại.


In [11]:
# ================== 04. SO SÁNH ĐIỂM CHECKPOINT ==================
reference = pd.read_csv(PATHS['baseline_06_5'], encoding='utf-8-sig')
reference['date'] = pd.to_datetime(reference['date'], format='%Y-%m-%d', errors='raise')
assert not reference.duplicated(['ticker', 'date']).any()
assert len(reference) == EXPECTED_VAL_SELECT_N
reference = select[['ticker', 'date']].merge(
    reference[['ticker', 'date', 'score_if', 'score_ae', 'score_lof']],
    on=['ticker', 'date'], how='left', validate='one_to_one'
)
assert reference[['score_if', 'score_ae', 'score_lof']].notna().all().all()

score_select = score_three(X_select)
TOLERANCES = {'IF': 1e-5, 'AE': 5e-4, 'LOF': 1e-5}
reproduction = []
for model in MODEL_NAMES:
    col = 'score_' + model.lower()
    recalculated = np.asarray(score_select[model], dtype=float)
    saved = reference[col].to_numpy(dtype=float)
    max_diff = float(np.max(np.abs(recalculated - saved)))
    reproduction.append({
        'model': model,
        'n_val_select': len(reference),
        'max_abs_score_diff': max_diff,
        'tolerance': TOLERANCES[model],
        'passed': bool(max_diff < TOLERANCES[model]),
    })
reproduction_df = pd.DataFrame(reproduction)
assert reproduction_df['passed'].all(), (
    'FAIL: Mô hình không tái lập kết quả 06.5 — không được hiệu chỉnh ngưỡng!\n'
    + reproduction_df.to_string(index=False)
)
print('PASS 04 — Tái lập checkpoint trên Validation-select:')
print(reproduction_df.to_string(index=False))

PASS 04 — Tái lập checkpoint trên Validation-select:
model  n_val_select  max_abs_score_diff  tolerance  passed
   IF          1430                 0.0    0.00001    True
   AE          1430                 0.0    0.00050    True
  LOF          1430                 0.0    0.00001    True


In [12]:
cal_scores = score_three(X_cal)
result = cal[['ticker','date',*FEATURES]].copy()
result['date'] = result.date.dt.strftime('%Y-%m-%d')
for model in MODEL_NAMES:
    scores = np.asarray(cal_scores[model],dtype=float)
    assert len(scores) == EXPECTED_CAL_N and np.isfinite(scores).all()
    result[f'score_{model.lower()}'] = scores

threshold_rows=[]
for model in MODEL_NAMES:
    s = result[f'score_{model.lower()}'].to_numpy(float)
    t = float(np.quantile(s,QUANTILE,method='linear'))
    assert np.isfinite(t)
    result[f'{model.lower()}_prediction'] = (s>t).astype('int8')
    threshold_rows.append({
        'model':model,
        'role':'main' if model=='IF' else 'comparison',
        'quantile':QUANTILE,
        'threshold':t,
        'rule':RULE,
        'n_calibration':len(s),
        'n_alert_calibration':int((s>t).sum()),
        'alert_rate_calibration_pct':100*float((s>t).mean()),
        'score_min':float(s.min()),
        'score_median':float(np.median(s)),
        'score_max':float(s.max()),
    })
thresholds_df = pd.DataFrame(threshold_rows)
result['n_models_alert'] = result[['if_prediction','ae_prediction','lof_prediction']].sum(axis=1).astype(int)
result['any_model_alert'] = (result.n_models_alert>0).astype('int8')
result['all_three_alert'] = (result.n_models_alert==3).astype('int8')
result['if_main_prediction'] = result['if_prediction']  # Chính thức của phương án A
assert set(result[['if_prediction','ae_prediction','lof_prediction']].stack().unique()) <= {0,1}
assert len(result)==EXPECTED_CAL_N and not result.duplicated(['ticker','date']).any()

print('NGƯỠNG CỦA PHƯƠNG ÁN A (chỉ từ Validation-calibrate):')
print(thresholds_df[['model','quantile','threshold','n_alert_calibration',
                     'alert_rate_calibration_pct']].to_string(index=False,float_format=lambda x:f'{x:.8f}'))

NGƯỠNG CỦA PHƯƠNG ÁN A (chỉ từ Validation-calibrate):
model   quantile  threshold  n_alert_calibration  alert_rate_calibration_pct
   IF 0.99000000 0.69630085                   10                  1.03950104
   AE 0.99000000 6.43437588                   10                  1.03950104
  LOF 0.99000000 1.51219421                   10                  1.03950104


In [13]:
patterns={
    (0,0,0):'none', (1,0,0):'IF_only', (0,1,0):'AE_only',
    (0,0,1):'LOF_only', (1,1,0):'IF_AE', (1,0,1):'IF_LOF',
    (0,1,1):'AE_LOF', (1,1,1):'all_three',
}
result['alert_pattern'] = [patterns[tuple(x)] for x in result[
    ['if_prediction','ae_prediction','lof_prediction']
].to_numpy(int)]
alert_summary = pd.DataFrame([
    {'pattern':label,'n_sessions':int((result.alert_pattern==label).sum())}
    for label in ['none','IF_only','AE_only','LOF_only','IF_AE','IF_LOF','AE_LOF','all_three']
])
assert alert_summary.n_sessions.sum() == EXPECTED_CAL_N

by_ticker = result.groupby('ticker',as_index=False).agg(
    n_sessions=('date','size'),
    n_if=('if_prediction','sum'),
    n_ae=('ae_prediction','sum'),
    n_lof=('lof_prediction','sum'),
    n_any=('any_model_alert','sum'),
    n_all_three=('all_three_alert','sum'),
)
# Ép kiểu float trước khi nhân 100 để tránh tràn số int8 (vd. 100*6 = 600).
by_ticker['if_rate_pct'] = (
    100.0 * by_ticker['n_if'].astype('float64') / by_ticker['n_sessions'].astype('float64')
)
assert by_ticker['if_rate_pct'].between(0,100).all(), 'Tỷ lệ cảnh báo theo mã nằm ngoài [0,100]'
assert np.allclose(
    by_ticker['if_rate_pct'].to_numpy(float),
    100.0 * by_ticker['n_if'].astype(float).to_numpy() / by_ticker['n_sessions'].astype(float).to_numpy()
)
assert by_ticker.n_sessions.eq(EXPECTED_CAL_DAYS).all()
assert by_ticker.n_if.sum() == int(result.if_prediction.sum())

by_date = result.groupby('date',as_index=False).agg(
    n_sessions=('ticker','size'),
    n_if=('if_prediction','sum'),
    n_ae=('ae_prediction','sum'),
    n_lof=('lof_prediction','sum'),
    n_any=('any_model_alert','sum'),
)
assert len(by_date) == EXPECTED_CAL_DAYS
assert by_date.n_sessions.eq(EXPECTED_TICKERS).all()

print('Phân bố mức đồng thuận (tập hiệu chỉnh, không phải đánh giá accuracy):')
print(alert_summary.to_string(index=False))
print('Số ngày có >=2 mã IF cùng cảnh báo:',int((by_date.n_if>=2).sum()))

Phân bố mức đồng thuận (tập hiệu chỉnh, không phải đánh giá accuracy):
  pattern  n_sessions
     none         946
  IF_only           3
  AE_only           2
 LOF_only           2
    IF_AE           1
   IF_LOF           1
   AE_LOF           2
all_three           5
Số ngày có >=2 mã IF cùng cảnh báo: 1


In [14]:
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
outputs={
    'scores':'07_calibration_scores.csv',
    'threshold_csv':'07_thresholds.csv',
    'threshold_json':'07_thresholds.json',
    'alerts':'07_calibration_alerts_any.csv',
    'patterns':'07_calibration_alert_patterns.csv',
    'by_ticker':'07_calibration_by_ticker.csv',
    'by_date':'07_calibration_by_date.csv',
    'reproduction':'07_checkpoint_reproduction.csv',
    'report':'07_calibration_QA_report.md',
    'config':'07_run_config.json',
}

result.to_csv(OUTPUT_DIR/outputs['scores'],index=False,encoding='utf-8-sig')
thresholds_df.to_csv(OUTPUT_DIR/outputs['threshold_csv'],index=False,encoding='utf-8-sig')
result[result.any_model_alert==1].to_csv(
    OUTPUT_DIR/outputs['alerts'],index=False,encoding='utf-8-sig'
)
alert_summary.to_csv(OUTPUT_DIR/outputs['patterns'],index=False,encoding='utf-8-sig')
by_ticker.to_csv(OUTPUT_DIR/outputs['by_ticker'],index=False,encoding='utf-8-sig')
by_date.to_csv(OUTPUT_DIR/outputs['by_date'],index=False,encoding='utf-8-sig')
reproduction_df.to_csv(OUTPUT_DIR/outputs['reproduction'],index=False,encoding='utf-8-sig')

thresholds_json={
    'step':'07_threshold_calibration_A',
    'method':'pooled_global_per_model_empirical_quantile',
    'quantile':QUANTILE,
    'numpy_quantile_method':'linear',
    'rule':RULE,
    'score_direction':'higher_is_more_anomalous',
    'calibration_split':'val',
    'calibration_val_role':'calibrate',
    'n_calibration':EXPECTED_CAL_N,
    'calibration_first_date':DATE_START,
    'calibration_last_date':DATE_END,
    'feature_cols':FEATURES,
    'feature_engineering':'W60, 1.4826*MAD + Train-only floor, Step03',
    'thresholds':{row['model']:{
        'threshold':row['threshold'],
        'role':row['role'],
        'score_column':f"score_{row['model'].lower()}",
        'prediction_column':f"{row['model'].lower()}_prediction",
        'n_calibration_alerts':row['n_alert_calibration'],
    } for row in threshold_rows},
    'model_sha256':{m:sha256_file(PATHS[m.lower()]) for m in MODEL_NAMES},
    'feature_config_sha256':sha256_file(PATHS['feature_config']),
    'feature_data_sha256':sha256_file(PATHS['features_all']),
    'source_for_calibration':'03_model_ready_development.csv only; full features file used for sha256 provenance only',
    'no_refit':True,
    'no_test_scoring':True,
    'no_proxy_news_or_actual_labels':True,
}
(OUTPUT_DIR/outputs['threshold_json']).write_text(
    json.dumps(thresholds_json,ensure_ascii=False,indent=2),encoding='utf-8'
)

config={
    'step':'07_threshold_calibration_A',
    'created_at_utc':datetime.now(timezone.utc).isoformat(),
    'root':str(ROOT.resolve()),
    'features':FEATURES,
    'W':60,
    'train_floors':feature_config['train_floors'],
    'quantile_predeclared':QUANTILE,
    'comparison_rule':RULE,
    'n_val_select_reproduction':EXPECTED_VAL_SELECT_N,
    'n_val_calibration':EXPECTED_CAL_N,
    'calibration_period':[DATE_START,DATE_END],
    'input_paths_and_sha256':{
        name:{'path':str(path.resolve()),'sha256':sha256_file(path)}
        for name,path in PATHS.items()
    },
    'model_reproduction':reproduction,
    'scikit_learn_version_current':sklearn.__version__,
    'torch_version_current':torch.__version__,
    'python_version_current':platform.python_version(),
    'scikit_learn_version_warnings':version_warnings,
    'no_training_or_threshold_tuning_on_calibration':True,
    'no_test_scored':True,
}
(OUTPUT_DIR/outputs['config']).write_text(
    json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8'
)

# Trình bày bằng to_string để notebook không cần thư viện tabulate.
report=[
    '# Bước 07 — Threshold Calibration (phương án A)',
    '',
    '## Quy tắc cố định',
    '- Đầu vào model: `z_gap`, `z_intraday`, `z_range`, `z_volume`.',
    '- Chỉ chấm Val-calibrate: 962 phiên, 13 mã × 74 ngày, 2025-09-11 đến 2025-12-23.',
    '- IF (mô hình chính), AE và LOF (đối chiếu). Không fit/refit.',
    '- Mỗi mô hình tính ngưỡng chung cho tất cả 13 mã: phân vị 99% của 962 điểm.',
    '- `score > threshold` => `prediction=1`; ngược lại 0. Không dùng `predict()` mặc định.',
    '- Chưa dùng Test, proxy hoặc news; chưa lựa chọn lại mô hình/ngưỡng.',
    '',
    '## Đối chiếu checkpoint trên Val-select',
    '```', reproduction_df.to_string(index=False), '```','',
    '## Ngưỡng và số cảnh báo trên Val-calibrate',
    '```',thresholds_df.to_string(index=False,float_format=lambda x:f'{x:.9g}'),'```','',
    '## Đồng thuận cảnh báo trong Val-calibrate',
    '```',alert_summary.to_string(index=False),'```','',
    '## Cảnh báo chuyên môn',
    '- Tỷ lệ 1% là mức ngân sách cảnh báo thiết kế, không phải tỷ lệ bất thường thật hoặc Accuracy/Recall.',
    '- Ngưỡng phân vị tính trên Validation hiệu chỉnh; không được quay lại tinh chỉnh IF/AE/LOF sau khi xem tập này.',
    '- Các ngày và mã có cảnh báo chỉ được diễn giải mô tả OHLCV; không gán sự kiện/news ở Thí nghiệm 1.',
    '- Ngưỡng theo toàn bộ mẫu (pooled). Một số mã có thể có nhiều cảnh báo hơn; đây không phải lỗi tự động.',
    '- Các mô hình sử dụng các thang điểm riêng; không so sánh trực tiếp trị số threshold giữa IF, AE và LOF.',
    '- File joblib khác phiên bản scikit-learn cần đối chiếu điểm Val-select; bước này đã đối chiếu thành công.',
    '',
    '## Tài liệu được giữ cho Bước 08',
    '- `07_thresholds.json`: ngưỡng đóng băng cho cả ba mô hình.',
    '- `07_calibration_scores.csv`: điểm và dự đoán trên Val-calibrate (không phải Test).',
    '- `07_run_config.json`: dấu vết đầu vào, checkpoint, thời gian và kiểm tra.',
    '',
]
(OUTPUT_DIR/outputs['report']).write_text('\n'.join(report),encoding='utf-8')

2647

In [15]:
roundtrip = pd.read_csv(OUTPUT_DIR/outputs['scores'],encoding='utf-8-sig')
loaded_thresholds = json.loads((OUTPUT_DIR/outputs['threshold_json']).read_text(encoding='utf-8'))
assert len(roundtrip) == EXPECTED_CAL_N
assert not roundtrip.duplicated(['ticker','date']).any()
assert roundtrip.date.min() == DATE_START and roundtrip.date.max() == DATE_END
assert set(roundtrip.columns) >= {'ticker','date',*FEATURES,'score_if','score_ae','score_lof',
                                'if_prediction','ae_prediction','lof_prediction'}
assert loaded_thresholds['calibration_split']=='val'
assert loaded_thresholds['calibration_val_role']=='calibrate'
assert loaded_thresholds['no_test_scoring'] is True
for model in MODEL_NAMES:
    key=model.lower()
    t=loaded_thresholds['thresholds'][model]['threshold']
    actual = roundtrip[f'{key}_prediction'].to_numpy(dtype=int)
    expected = (roundtrip[f'score_{key}'].to_numpy(float)>t).astype(int)
    assert np.array_equal(actual,expected),f'Kết quả {model} không khớp ngưỡng đã lưu'
    assert int(actual.sum()) == loaded_thresholds['thresholds'][model]['n_calibration_alerts']
assert len(roundtrip) == sum(1 for _ in range(len(cal)))

print('\n'+'='*62)
print('PASS: BƯỚC 07 HIỆU CHỈNH NGƯỠNG PHƯƠNG ÁN A HOÀN TẤT')
print('Đã tính ngưỡng cho IF / AE / LOF, không huấn luyện lại.')
print('Không chấm Test. Chưa có đánh giá accuracy thực tế.')
print('Đường dẫn:',OUTPUT_DIR.resolve())
print('='*62)


PASS: BƯỚC 07 HIỆU CHỈNH NGƯỠNG PHƯƠNG ÁN A HOÀN TẤT
Đã tính ngưỡng cho IF / AE / LOF, không huấn luyện lại.
Không chấm Test. Chưa có đánh giá accuracy thực tế.
Đường dẫn: D:\LUẬN VĂN\stock-news-anomaly-project\Stocks\notebooks\outputs\07_threshold_calibration_A
